In [1]:
import numpy as np
from pathlib import Path
import shutil
import random
import cv2
from sklearn.cluster import KMeans
from collections import defaultdict

# ============================================
# НАСТРОЙКИ
# ============================================

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

DATA_PATH = Path("data/256_yolo/clean_patches")
OUTPUT_PATH = Path("data/256_yolo/balanced_clean_patches")
TARGET_COUNT = 2000

# Соотношения для разбиения
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15

print("="*80)
print("ОТБОР ЧИСТЫХ ПАТЧЕЙ С РАЗБИЕНИЕМ 70/15/15")
print("="*80)

# ============================================
# 1. ПОИСК ВСЕХ ИЗОБРАЖЕНИЙ
# ============================================

print(f"\n📁 ПОИСК ИЗОБРАЖЕНИЙ:")

all_images = []
for ext in ['*.png', '*.jpg', '*.jpeg']:
    all_images.extend(DATA_PATH.rglob(ext))

print(f"  Найдено изображений: {len(all_images)}")

if len(all_images) == 0:
    print("  ❌ Изображения не найдены!")
    exit(1)

if len(all_images) <= TARGET_COUNT:
    print(f"  ⚠️ Изображений меньше {TARGET_COUNT}, будут отобраны все")
    selected_images = all_images
else:
    # ============================================
    # 2. АНАЛИЗ И КЛАСТЕРИЗАЦИЯ
    # ============================================
    
    print(f"\n📊 АНАЛИЗ ИЗОБРАЖЕНИЙ:")
    
    features = []
    valid_images = []
    sample_size = min(5000, len(all_images))
    sample_images = random.sample(all_images, sample_size)
    
    for img_path in sample_images:
        img = cv2.imread(str(img_path))
        if img is not None:
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            mean_val = np.mean(gray)
            std_val = np.std(gray)
            hist = cv2.calcHist([gray], [0], None, [32], [0, 256])
            hist = hist.flatten() / hist.sum()
            feat = np.concatenate([[mean_val, std_val], hist[:10]])
            features.append(feat)
            valid_images.append(img_path)
    
    features = np.array(features)
    print(f"  Проанализировано: {len(features)} изображений")
    
    print(f"\n🔍 КЛАСТЕРИЗАЦИЯ:")
    n_clusters = 20
    kmeans = KMeans(n_clusters=n_clusters, random_state=RANDOM_SEED, n_init=10)
    clusters = kmeans.fit_predict(features)
    
    cluster_images = defaultdict(list)
    for img_path, cluster in zip(valid_images, clusters):
        cluster_images[cluster].append(img_path)
    
    print(f"  Кластеров: {n_clusters}")
    
    # ============================================
    # 3. РАВНОМЕРНЫЙ ОТБОР
    # ============================================
    
    print(f"\n📦 ОТБОР {TARGET_COUNT} ИЗОБРАЖЕНИЙ:")
    
    per_cluster = TARGET_COUNT // n_clusters
    remainder = TARGET_COUNT % n_clusters
    
    selected_images = []
    
    for cluster_id in range(n_clusters):
        available = cluster_images[cluster_id]
        to_select = per_cluster + (1 if cluster_id < remainder else 0)
        to_select = min(to_select, len(available))
        selected = random.sample(available, to_select)
        selected_images.extend(selected)
    
    print(f"  Отобрано: {len(selected_images)}")

# ============================================
# 4. РАЗБИЕНИЕ НА TRAIN/VAL/TEST
# ============================================

print(f"\n📊 РАЗБИЕНИЕ 70/15/15:")

random.shuffle(selected_images)

total = len(selected_images)
train_end = int(total * TRAIN_RATIO)
val_end = train_end + int(total * VAL_RATIO)

splits = {
    'train': selected_images[:train_end],
    'val': selected_images[train_end:val_end],
    'test': selected_images[val_end:]
}

for split_name, split_images in splits.items():
    print(f"  {split_name}: {len(split_images)} изображений")

# ============================================
# 5. КОПИРОВАНИЕ
# ============================================

print(f"\n📁 КОПИРОВАНИЕ:")

if OUTPUT_PATH.exists():
    shutil.rmtree(OUTPUT_PATH)

for split_name in ['train', 'val', 'test']:
    (OUTPUT_PATH / split_name).mkdir(parents=True)

copied = 0
for split_name, split_images in splits.items():
    for img_path in split_images:
        dst_path = OUTPUT_PATH / split_name / img_path.name
        if dst_path.exists():
            dst_path = OUTPUT_PATH / split_name / f"{img_path.stem}_{copied}{img_path.suffix}"
        shutil.copy2(img_path, dst_path)
        copied += 1

print(f"  ✅ Скопировано: {copied} изображений")

# ============================================
# 6. ИТОГ
# ============================================

print("\n" + "="*80)
print("ГОТОВО!")
print("="*80)
print(f"""
✅ ОТОБРАНО: {total} чистых патчей

📁 ВЫХОДНАЯ ПАПКА: {OUTPUT_PATH}
   ├── train/ ({len(splits['train'])} изображений)
   ├── val/   ({len(splits['val'])} изображений)
   └── test/  ({len(splits['test'])} изображений)

📊 РАЗБИЕНИЕ: {TRAIN_RATIO*100:.0f}/{VAL_RATIO*100:.0f}/{TEST_RATIO*100:.0f}
""")

ОТБОР ЧИСТЫХ ПАТЧЕЙ С РАЗБИЕНИЕМ 70/15/15

📁 ПОИСК ИЗОБРАЖЕНИЙ:
  Найдено изображений: 34077

📊 АНАЛИЗ ИЗОБРАЖЕНИЙ:
  Проанализировано: 5000 изображений

🔍 КЛАСТЕРИЗАЦИЯ:
  Кластеров: 20

📦 ОТБОР 2000 ИЗОБРАЖЕНИЙ:
  Отобрано: 1766

📊 РАЗБИЕНИЕ 70/15/15:
  train: 1236 изображений
  val: 264 изображений
  test: 266 изображений

📁 КОПИРОВАНИЕ:
  ✅ Скопировано: 1766 изображений

ГОТОВО!

✅ ОТОБРАНО: 1766 чистых патчей

📁 ВЫХОДНАЯ ПАПКА: data/256_yolo/balanced_clean_patches
   ├── train/ (1236 изображений)
   ├── val/   (264 изображений)
   └── test/  (266 изображений)

📊 РАЗБИЕНИЕ: 70/15/15

